<center>
<img src="https://www.infnet.edu.br/infnet/wp-content/uploads/sites/18/2021/10/infnet-30-horizontal-padrao@300x-8-1024x265.png" width="60%"/>
</center>

# MBA em Engenharia de Dados: Big Data e IA
## Processamento de Big Data com Apache Spark e Spark SQL [26E3_2]



## Apache Spark (Continuação)

In [0]:
import os

import pyspark.sql.functions as F

In [0]:
CATALOGO = 'spark_26e3_2'
VOLUME = 'olist'
BASE_PATH = f'/Volumes/{CATALOGO}/{VOLUME}/raw/'


In [0]:
lista_arquivos = [os.path.join(BASE_PATH, file) for file in os.listdir(BASE_PATH) if file.endswith('.csv')]
lista_arquivos


In [0]:
def read_csv(file_path):
  return spark.read.csv(file_path, header=True, inferSchema=True)


In [0]:
olist_customers_df = read_csv(lista_arquivos[0])
olist_geolocation_df = read_csv(lista_arquivos[1])
olist_order_items_df = read_csv(lista_arquivos[2])
olist_order_payments_df = read_csv(lista_arquivos[3])
olist_order_reviews_df = read_csv(lista_arquivos[4])
olist_orders_df = read_csv(lista_arquivos[5])
olist_products_df = read_csv(lista_arquivos[6])
olist_sellers_df = read_csv(lista_arquivos[7])


## Tipos Básicos (Primitive Types)

| Data Type (PySpark) | Instanciação na API Python | Tipo equivalente em Python | Descrição / Faixa de Valores |
| :--- | :--- | :--- | :--- |
| **StringType** | `StringType()` | `str` | Cadeias de caracteres / texto UTF-8. |
| **IntegerType** | `IntegerType()` | `int` | Números inteiros de 32 bits (-2.147.483.648 a 2.147.483.647). |
| **LongType** | `LongType()` | `int` | Números inteiros de 64 bits (Bigint). |
| **ShortType** | `ShortType()` | `int` | Números inteiros de 16 bits (-32.768 a 32.767). |
| **ByteType** | `ByteType()` | `int` | Números inteiros de 8 bits (-128 a 127). |
| **FloatType** | `FloatType()` | `float` | Números de ponto flutuante de precisão simples (32 bits). |
| **DoubleType** | `DoubleType()` | `float` | Números de ponto flutuante de precisão dupla (64 bits). |
| **DecimalType** | `DecimalType(precision, scale)` | `decimal.Decimal` | Números decimais exatos (útil para dados financeiros). Ex: `DecimalType(10, 2)`. |
| **BooleanType** | `BooleanType()` | `bool` | Valores booleanos (`True` ou `False`). |
| **DateType** | `DateType()` | `datetime.date` | Datas no formato Ano-Mês-Dia (`YYYY-MM-DD`). |
| **TimestampType** | `TimestampType()` | `datetime.datetime` | Data e hora com fuso horário / milissegundos. |
| **TimestampNTZType** | `TimestampNTZType()` | `datetime.datetime` | Timestamp sem fuso horário (*No Time Zone*). |
| **BinaryType** | `BinaryType()` | `bytearray` / `bytes` | Sequência de bytes brutos (binários). |
| **NullType** | `NullType()` | `NoneType` (`None`) | Represeta um valor nulo / ausente sem tipo específico. |


## Tipos Complexos (Complex / Structural Types)

| Data Type (PySpark) | Value assigned in Python API to instantiate | Tipo equivalente em Python | Descrição / Exemplo |
| :--- | :--- | :--- | :--- |
| **ArrayType** | `ArrayType(elementType, [containsNull])` | `list`, `tuple` ou `array` | Representa uma coleção/lista de elementos do mesmo tipo. Por padrão `containsNull=True`. |
| **MapType** | `MapType(keyType, valueType, [valueContainsNull])` | `dict` | Representa um mapa de chaves e valores (*key-value*). Por padrão `valueContainsNull=True`. |
| **StructType** | `StructType(fields)` | `tuple`, `dict`, `pyspark.sql.Row` | Representa uma estrutura de dados com campos nomeados (similar a um objeto ou struct). Recebe uma lista de `StructField`. |
| **StructField** | `StructField(name, dataType, [nullable], [metadata])` | N/A | Define um campo individual dentro de um `StructType`, especificando nome, tipo e se aceita nulos. |

In [0]:
import pyspark.sql.types as T

In [0]:
# Mudando tipo coluna
olist_order_items_df.limit(5).display()

In [0]:
olist_order_items_df.printSchema()

In [0]:
olist_order_items_df = olist_order_items_df.withColumn(
    'shipping_limit_date',
    F.to_timestamp(F.col('shipping_limit_date'), 'yyyy-MM-dd HH:mm:ss')
)

olist_order_items_df.display()

In [0]:

olist_order_items_df = olist_order_items_df.withColumn(
    'price',
    F.col('price').cast(T.FloatType())
).withColumn(
    'freight_value',
    F.col('freight_value').cast('float')
)

olist_order_items_df.printSchema()


## Escrita de dados


### Formatos

- `PARQUET`
    - Fonte de dados padrão do Spark, e é altamente utilizado no contexto de Big Data por ser um formato muito eficiente e versátil.
- `JSON`
    - O formato JSON é também bastante popular e se faz presente em diversos contextos e aplicações, pois é o resultado de uma consulta à uma API.

- `CSV`
    - Arquivos CSV são uma das formas mais comuns de se compartilhar e administrar dados. Nesses arquivos, os dados são organizados de forma tabular e o valor de cada uma das colunas é separado por um delimitador, usualmente uma vírgula.

- `ORC`
    - No formato ORC, semelhante ao parquet, os dados são armazenados de forma colunar objetivando alcançar maior eficiência. Desenvolvido para cargas de trabalho Hadoop, os arquivos ORC também podem ser lidos no Spark a partir da versão 2.0. A principal diferença entre o uso de arquivos ORC e arquivos parquet é que o Spark implementa otimizações específicas para o uso do segundo, o que o torna preferível.





### Modo

A principal configuração da escrita é o `mode`, argumento que indica qual o comportamento do Spark caso ele encontre dados já existentes no diretório indicado como destino dos dados. As opções são as seguintes:
- `append`: anexa o conteúdo do DataFrame aos dados existentes.
- `overwrite`: sobrescreve dados existentes.
- `ignore`: ignora silenciosamente essa operação se os dados já existirem.
- `error` ou `errorifexists` (default): retorna erro se os dados já existirem.

In [0]:
from pathlib import Path

OUTPUT_PATH = Path(BASE_PATH, 'parquet')
OUTPUT_PATH.mkdir(exist_ok=True)

comment_messages_not_null = (
    olist_order_reviews_df
    .filter(
        F.col('review_comment_message').isNotNull()
    )
    .select(
        'review_id',
        'order_id',
        'review_comment_message'
    )
)

(
    comment_messages_not_null
    .write
    .mode('overwrite')
    .parquet(str(Path(OUTPUT_PATH, 'comment_messages_not_null')))
)


In [0]:
# A funcao alias

comment_messages = (
    olist_order_reviews_df
    .filter(
        F.col('review_comment_message').isNotNull()
    )
    .select(
        F.col('review_id'),
        F.col('order_id'),
        F.col('review_comment_message').alias('comment_message')
    )
)

(
    comment_messages
    .coalesce(1)
    .write
    .mode('overwrite')
    .parquet(str(Path(OUTPUT_PATH, 'comment_messages')))
)


In [0]:
# Ordenacao

(
    olist_order_items_df
    .select(
        'order_id',
        'price',
    )
    .groupBy('order_id')
    .agg(
        F.round(F.sum('price'), 2).alias('total_price')
    )
    .orderBy(F.col('total_price').desc())
    .display()
)


In [0]:
olist_order_reviews_df.printSchema()

In [0]:
# Valores nulos
# drop() any ou all
# fill()
# replace()
# filna()
# para especificar um conjuntos de colunas, utiliza-se o atributo subset
olist_order_reviews_df.fillna(
    'sem texto',
    subset=['review_comment_title', 'review_comment_message']
).display()


## Trabalhando com Números


Quando se trabalha com valores numéricos, as funções mais utilizadas estão principalmente relacionadas às transformações matemáticas que podem ser aplicadas sobre esses valores. Vale destacar também algumas funções úteis para comparação, como encontrar o maior ou menor valor em um conjunto. 

Abaixo uma lista das funções mais usadas: 
- `rand()`: retorna uma amostra independente de uma distribuição uniforme entre 0 e 1. 
- `randn()`: retorna uma amostra independente de uma distribuição normal padrão (média 0 e variância 1). 
- `round()`: arredonda o valor. 
- `ceil()`: arredonda o valor para o maior inteiro mais próximo. 
- `floor()`: arredonda o valor para o menor inteiro mais próximo. 
- `sqrt()`: retorna a raiz quadrada do valor. 
- `exp()`: retorna a exponencial do valor. 
- `log()`: retorna a logaritmo natural do valor. 
- `log10()`: retorna a logaritmo na base 10 do valor. 
- `pow()`: retorna o valor de uma coluna elevado a potência passada pelo usuário.


## Trabalhando com Strings


A tarefa mais importante ao lidar com strings é formatá-los de forma que eles sigam algum padrão estabelecido, e contenham somente as informações necessárias. Isto é, os caracteres devem ser transformados, e muitas vezes removidos. As funções a seguir auxiliam nesse propósito:

- `upper()`: retorna o string em letras maiúsculas. 
- `lower()`: retorna o string em letras minúsculas. 
- `trim()`: retira os espaços em branco do início e do final do string. 
- `lpad()` / `rpad()`: acrescenta um caractere no início e no final do string, respectivamente, até que o string tenha um determinado comprimento. 
- `length()`: retorna o comprimento do string, em quantidade de caracteres. 
- `split()`: quebra o string a partir de um padrão e retorna um array com os string resultantes. 
- `concat()`: concatena uma ou mais colunas de string.
- `concat_ws()`: concatena uma ou mais colunas de string, com um separador entre elas. 
- `regexp_extract()`: retorna um match no string a partir de um padrão regex. 
- `regexp_replace()`: substitui um match no string a partir de um padrão regex com outros caracteres passados para a função. 
- substring(): retorna os caracteres do string que estão entre os índices especificados. Análogo a `col().substring()`.


## Trabalhando com Datas


Trabalhar com datas é um desafio constante para profissionais da área de dados. Cada lugar do mundo usa um diferente padrão de armazenamento de dados, sem contar que diferentes ferramentas usam diferentes formas de armazenar e computar datas.

Para lidar com todas essas peculiaridades, o Spark implementa algumas funções bastante úteis para manipular campos de data e tempo: 

- `add_months()`: retorna a data depois de adicionar "x" meses. 
- `months_between()`: retorna a diferença entre duas datas em meses. 
- `date_add()`: retorna a data depois de adicionar "x" dias. 
- `date_sub()`: retorna a data depois de subtrair "x" dias. 
- `next_day()`: retorna o dia seguinte de alguma data. 
- `datediff()`: retorna a diferença entre duas datas em dias. 
- `current_date()`: retorna a data atual. 
- `dayofweek()` / `dayofmonth()` / `dayofyear()`: retorna o dia relativo à semana, ao mês e ao ano, respectivamente. 
- `weekofyear()`: retorna a semana relativa ao ano. 
- `second()` / `minute()` / `hour()`: retorna os segundos, os minutos e as horas de uma coluna de datetime, respectivamente.
- `month()` / `year()`: retorna o mês e o ano de uma coluna de data, respectivamente. 
- `to_date()`: transforma a coluna no tipo data (`DateType()`). 




## Mais sobre Agrupamento e Agregação

No Spark, é possível realizar agregação utilizando o método `agg()`, que recebe uma especificação de coluna ou expressão que retorne um valor escalar (o que caracteriza uma função de agregação).
As principais funções de agregação:

- `sum()`: retorna a soma dos valores da coluna; 
- sumDistinct()`: retorna a soma dos valores distintos da coluna;
- `min() / max()`: retorna o mínimo e o máximo da coluna, respectivamente; 
- `avg() / mean()`: retorna a média dos valores da coluna; 
- `percentile_approx()`: retorna o percentil da coluna, com aproximação. Para trazer a mediana exata, usar: `percentile_approx(col('col1'), 0.5, lit(1000000))`; 
- `stddev()`: retorna o desvio padrão dos valores da coluna; 
- `count()`: retorna a contagem de linhas; 
- `countDistinct()`: retorna a contagem de valores distintos da coluna; 
- `first() / last()`: retorna o primeiro e o último valor da coluna, respectivamente. Interessante de ser utilizada em conjunto com o argumento ignoreNulls=True; 
- `collect_list()`: retorna os valores da coluna em uma lista, com duplicações; 
- `collect_set()`: retorna os valores da coluna em uma lista, sem duplicações (desordenado);
- `expr()`: é possível usar a função de criação de expressões, desde que seja denotada uma operação de agregação por meio dela.



Além de `agg(`) e `count()`, existem ainda outros métodos capazes de realizar agregações. Alguns deles são: 
- `describe()`: computa diversas estatísticas básicas para todas as colunas numéricas ou de strings no DataFrame. Essas quantidades são a contagem, média, desvio padrão, mínimo e máximo; 
- `approxQuantile()`: calcula um ou mais quantis das colunas numéricas de um DataFrame; 
- `corr()`: calcula a correlação de Pearson entre duas colunas numéricas de um DataFrame.


## Window Functions


São funções que realizam cálculos similares a uma agregação, mas que não resultam em um DataFrame agregado. Ao invés disso, os resultados são colocados em uma nova coluna, segundo a partição de linhas (ou agrupamento) especificada.

O Spark suporta três tipos de window functions: funções de agregação, funções de ranqueamento e funções analíticas.

A forma padrão de uma janela segue: 
```
    Window.partitionBy({columns})
        .orderBy({columns}) 
        .rowsBetween({lower}, {upper})
```

Em que cada um dos métodos é opcional para a definição da janela. Sobre esses elementos:
- `partitionBy()`: agrupamento em que os cálculos serão realizados. É análogo ao `groupBy()`. 
- `orderBy`: funções como `row_number()` e `lag()` dependem da ordenação das linhas do agrupamento. Essa função é usada para especificar a ordem desejada. 
`rowsBetween()`: esse método é usado para especificar janelas deslizantes.




## Joins


Em manipulação de dados, os joins (ou junção, em português) são uma classe de operações de banco de dados usada para relacionar duas tabelas com base em colunas comuns entre elas. Os joins no pyspark são especificados pela função `join()`, da seguinte forma:

```
df1.join(df2, {colunas_chave}, {tipo_join})
```


### Escolhendo o melhor tipo de Join


No Spark, é possível escolher diferentes tipos de algoritmo para a realização de joins, a fim de escolher os mais adequados para cada situação. Existem pelo menos três tipos de algoritmos para joins, sendo eles:
- Broadcast Join (BHJ): é o algoritmo de join mais eficiente do Spark, mas só pode ser usado em situações específicas. Ele requer que um dos DataFrames seja pequeno o suficiente para caber na memória do driver e de cada um dos executores, pois a estratégia consiste em enviar esses dados completos para cada um deles, de forma que só há necessidade de realizar o shuffle de envio desses dados.
- Shuffled Hash Join (SHJ): é o algoritmo padrão do Spark, uma vez que o tamanho dos DataFrames não impacta na viabilidade do algoritmo. Nesse caso, os dados são enviados entre os executores via shuffle e os posteriormente ordenados, para que os dados estejam particionados corretamente e na mesma ordem.
- Sort Merge Join (SMJ): é um algoritmo que também usa shuffles, mas compensa essa operação com o uso de um mapa de hash que exime a necessidade de ordenação dos dados. A única condição é que um dos DataFrames seja significativamente menor do que o outro, mas não tanto quanto o BHJ.


## Unions


Uma outra forma de unir a informação de duas tabelas é simplesmente empilhar as duas, tornando-as em uma só. Para realizar essa operação, é necessário garantir que os dois DataFrames tenham o mesmo número de colunas e o mesmo schema (ou seja, tipos das colunas), e a operação é feita utilizando um de dois métodos: `union()` ou `unionByName()`.

No primeiro, as colunas são concatenadas com base na sua posição no DataFrame, um comportamento que pode levar a erros inesperados e difíceis de depurar. O segundo método, por sua vez, resolve esse problema ao concatenar colunas pelo nome, de forma que o usuário só precisa garantir que as colunas que devam ser concatenadas tenham o mesmo nome.